# LeetCode 1282 — Group the People Given the Group Size They Belong To bằng Python

Link đề: https://leetcode.com/problems/group-the-people-given-the-group-size-they-belong-to/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Greedy

Skeleton LeetCode:

```python
class Solution:
    def groupThePeople(self, groupSizes: list[int]) -> list[list[int]]:
        ...
```

> Bài này **có nhiều đáp án đúng** (thứ tự nhóm và cách chia đều tuỳ ý). Vì vậy trong notebook ta kiểm tra kết quả bằng **hàm kiểm tra tính hợp lệ**, không so sánh trực tiếp với output mẫu.

## 1. Mục tiêu học

1. Biết dùng **dict / `defaultdict(list)`** để gom các phần tử theo một khoá (ở đây là kích thước nhóm) — kỹ thuật "bucket".
2. Hiểu chiến lược **tham lam "đổ đầy rồi chốt"**: thêm người vào nhóm đang mở, đủ người thì đóng nhóm và mở nhóm mới.
3. Đi từ lời giải ngây thơ `O(n²)` lên `O(n)` và một lời giải `O(n log n)` bằng sắp xếp.
4. Biết **kiểm thử bài nhiều đáp án**: viết hàm `is_valid` thay vì `assert output == expected`.
5. Tránh các lỗi: quên cắt bucket, **dùng chung một list** khi chốt nhóm (`append` rồi `clear`), so sánh cứng với output mẫu.

## 2. Đề bài & ví dụ

Có `n` người, mỗi người có ID duy nhất từ `0` đến `n-1`. Mảng `groupSizes` cho biết `groupSizes[i]` là **kích thước nhóm** mà người `i` thuộc về. Hãy trả về một danh sách các nhóm sao cho:

- mỗi người `i` ở trong một nhóm có đúng `groupSizes[i]` người;
- mỗi người xuất hiện **đúng một lần**.

Nếu có nhiều đáp án, trả về bất kỳ đáp án nào. Đề đảm bảo luôn có ít nhất một đáp án.

**Ví dụ 1:** `groupSizes = [3,3,3,3,3,1,3]` → ví dụ `[[5],[0,1,2],[3,4,6]]`. Đáp án khác cũng đúng: `[[2,1,6],[5],[0,4,3]]`, `[[5],[0,6,2],[4,3,1]]`.

**Ví dụ 2:** `groupSizes = [2,1,3,3,3,2]` → ví dụ `[[1],[0,5],[2,3,4]]`.

**Ràng buộc:** `1 <= n <= 500`, `1 <= groupSizes[i] <= n`.

In [1]:
import random
import time
from collections import defaultdict, Counter
from math import factorial
from typing import List

examples = [
    ([3, 3, 3, 3, 3, 1, 3], [[5], [0, 1, 2], [3, 4, 6]]),
    ([2, 1, 3, 3, 3, 2], [[1], [0, 5], [2, 3, 4]]),
]
for gs, sample in examples:
    print("groupSizes =", gs, "-> một đáp án mẫu:", sample)

groupSizes = [3, 3, 3, 3, 3, 1, 3] -> một đáp án mẫu: [[5], [0, 1, 2], [3, 4, 6]]
groupSizes = [2, 1, 3, 3, 3, 2] -> một đáp án mẫu: [[1], [0, 5], [2, 3, 4]]


## 3. Kiến thức nền

### 3.1 `enumerate` — duyệt cả chỉ số và giá trị

Ở bài này, **chỉ số** chính là ID người, **giá trị** là kích thước nhóm.

In [2]:
gs = [2, 1, 3, 3, 3, 2]
for person, size in enumerate(gs):
    print(f"người {person} cần nhóm {size} người")

người 0 cần nhóm 2 người
người 1 cần nhóm 1 người
người 2 cần nhóm 3 người
người 3 cần nhóm 3 người
người 4 cần nhóm 3 người
người 5 cần nhóm 2 người


### 3.2 `defaultdict(list)` — gom theo khoá

Truy cập khoá chưa có sẽ tự tạo `[]`, nên có thể `append` ngay mà không cần `if key not in d`.

In [3]:
buckets = defaultdict(list)
for person, size in enumerate(gs):
    buckets[size].append(person)
print(dict(buckets))

{2: [0, 5], 1: [1], 3: [2, 3, 4]}


### 3.3 Cắt list thành các đoạn bằng nhau

`lst[i:i+k]` lấy đoạn độ dài `k` bắt đầu ở `i`. `range(0, len(lst), k)` cho các điểm bắt đầu `0, k, 2k, …`.

In [4]:
lst = [0, 1, 2, 3, 4, 6]
k = 3
print([lst[i:i + k] for i in range(0, len(lst), k)])

[[0, 1, 2], [3, 4, 6]]


### 3.4 Tham chiếu list: `append(cur)` rồi `cur.clear()` là nguy hiểm

`res.append(cur)` chỉ thêm **tham chiếu** tới cùng một list. Nếu sau đó `cur.clear()`, list trong `res` cũng bị xoá. Muốn "mở nhóm mới" phải gán `cur = []` (tạo list mới).

In [5]:
res = []
cur = [1, 2]
res.append(cur)
cur.clear()                      # xoá luôn phần tử trong res!
print("sau clear():", res)

res = []
cur = [1, 2]
res.append(cur)
cur = []                         # cur trỏ sang list mới, res giữ list cũ
print("sau cur = []:", res)

sau clear(): [[]]
sau cur = []: [[1, 2]]


### 3.5 `sorted` với `key` và `Counter`

`sorted(range(n), key=lambda i: gs[i])` sắp xếp các ID theo kích thước nhóm. `Counter` đếm tần suất — hữu ích để kiểm tra "số người cần nhóm cỡ `s`" có chia hết cho `s` không.

In [6]:
ids = sorted(range(len(gs)), key=lambda i: gs[i])
print("ID sắp theo kích thước:", ids, "->", [gs[i] for i in ids])
cnt = Counter(gs)
print("đếm:", cnt, "| chia hết?", all(c % s == 0 for s, c in cnt.items()))

ID sắp theo kích thước: [1, 0, 5, 2, 3, 4] -> [1, 2, 2, 3, 3, 3]
đếm: Counter({3: 3, 2: 2, 1: 1}) | chia hết? True


## 4. Ý tưởng nền tảng / trực giác

Làm tay **Ví dụ 2** `[2, 1, 3, 3, 3, 2]`:

- Người cần nhóm cỡ 1: `{1}` ⇒ nhóm `[1]`.
- Người cần nhóm cỡ 2: `{0, 5}` ⇒ đúng 2 người ⇒ nhóm `[0, 5]`.
- Người cần nhóm cỡ 3: `{2, 3, 4}` ⇒ nhóm `[2, 3, 4]`.

Với **Ví dụ 1**, có 6 người cần nhóm cỡ 3: `{0,1,2,3,4,6}` ⇒ chia thành **hai** nhóm 3 người bất kỳ, ví dụ `[0,1,2]` và `[3,4,6]`.

Nhận xét then chốt: người cần nhóm cỡ `s` **chỉ có thể** ở cùng nhóm với người cũng cần nhóm cỡ `s`. Vậy các kích thước khác nhau là **hoàn toàn độc lập**; trong cùng một kích thước, cứ lấy lần lượt `s` người là được.

In [7]:
def is_valid(groupSizes, groups):
    """Kiểm tra một đáp án: mỗi người đúng 1 lần và nhóm có đúng kích thước yêu cầu."""
    n = len(groupSizes)
    seen = [0] * n
    for g in groups:
        for p in g:
            if not (0 <= p < n):
                return False
            seen[p] += 1
            if groupSizes[p] != len(g):
                return False
    return all(c == 1 for c in seen)

for gs, sample in examples:
    print(gs, "| đáp án mẫu hợp lệ?", is_valid(gs, sample))
print("đáp án khác của Ví dụ 1:", is_valid(examples[0][0], [[2, 1, 6], [5], [0, 4, 3]]))
print("đáp án sai (nhóm 4 người):", is_valid(examples[0][0], [[5], [0, 1, 2, 3], [4, 6]]))

[3, 3, 3, 3, 3, 1, 3] | đáp án mẫu hợp lệ? True
[2, 1, 3, 3, 3, 2] | đáp án mẫu hợp lệ? True
đáp án khác của Ví dụ 1: True
đáp án sai (nhóm 4 người): False


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng ngây thơ:** duyệt từng người `i` chưa được xếp. Tạo nhóm mới chứa `i`, rồi **quét lại cả mảng** tìm thêm người chưa xếp có cùng kích thước cho đến khi đủ `groupSizes[i]` người.

In [8]:
def group_brute(groupSizes):
    n = len(groupSizes)
    used = [False] * n
    res = []
    for i in range(n):
        if used[i]:
            continue
        s = groupSizes[i]
        group = []
        for j in range(i, n):                 # quét tìm người cùng kích thước
            if not used[j] and groupSizes[j] == s:
                group.append(j)
                used[j] = True
                if len(group) == s:
                    break
        res.append(group)
    return res

for gs, _ in examples:
    out = group_brute(gs)
    print(out, "| hợp lệ:", is_valid(gs, out))

[[0, 1, 2], [3, 4, 6], [5]] | hợp lệ: True
[[0, 5], [1], [2, 3, 4]] | hợp lệ: True


**Độ phức tạp:** mỗi lần mở nhóm quét tới `O(n)` phần tử, có tới `O(n)` nhóm (khi toàn kích thước 1… thực ra khi đó quét dừng sớm; worst là nhiều kích thước xen kẽ) ⇒ `O(n²)`. Với `n ≤ 500` vẫn chạy được, nhưng ta làm tốt hơn dễ dàng.

**Nút thắt:** quét lại mảng để "tìm người cùng kích thước" — việc này có thể làm **một lần** bằng cách gom sẵn theo kích thước.

In [9]:
# Worst case cho brute force: người cùng kích thước nằm rải rác, nhóm cỡ lớn
n = 500
gs_bad = [250] * 250 + [1] * 250
random.Random(0).shuffle(gs_bad)
t = time.perf_counter()
out = group_brute(gs_bad)
print("hợp lệ:", is_valid(gs_bad, out), f"| {time.perf_counter() - t:.4f}s")

hợp lệ: True | 0.0002s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** gom người theo kích thước nhóm. Với kích thước `s`, có `c_s` người; do đề đảm bảo có đáp án nên `c_s` chia hết cho `s`. Cắt danh sách thành `c_s / s` đoạn liên tiếp, mỗi đoạn `s` người.

**Phiên bản tham lam "đổ đầy rồi chốt":** giữ cho mỗi kích thước `s` một **nhóm đang mở**. Gặp người `i` có kích thước `s`: thêm vào nhóm mở của `s`; nếu nhóm đủ `s` người thì chốt nhóm (đưa vào kết quả) và mở nhóm mới.

**Quy tắc "nếu… thì…":**
- Nếu `len(open[s]) == s` ⇒ chốt nhóm, `open[s] = []`.
- Nếu `s == 1` ⇒ người đó tự thành nhóm ngay.
- Kết thúc duyệt ⇒ mọi nhóm mở đều rỗng (vì `c_s` chia hết cho `s`).

**Invariant:** tại mọi thời điểm, `open[s]` có **ít hơn** `s` người, và mọi nhóm đã chốt có đúng `s` người cùng kích thước `s`.

## 7. Các cách tiếp theo

### Cách 2 — Gom bucket rồi cắt

**Từng đoạn:**
1. `buckets[s]` = danh sách người cần nhóm cỡ `s`.
2. Với mỗi `(s, people)`: cắt `people[i:i+s]` với `i = 0, s, 2s, …`.

In [10]:
def group_bucket(groupSizes):
    buckets = defaultdict(list)
    for person, size in enumerate(groupSizes):
        buckets[size].append(person)
    res = []
    for size, people in buckets.items():
        for i in range(0, len(people), size):
            res.append(people[i:i + size])
    return res

for gs, _ in examples:
    out = group_bucket(gs)
    print(out, "| hợp lệ:", is_valid(gs, out))

[[0, 1, 2], [3, 4, 6], [5]] | hợp lệ: True
[[0, 5], [1], [2, 3, 4]] | hợp lệ: True


In [11]:
class SolutionBucket:
    def groupThePeople(self, groupSizes: List[int]) -> List[List[int]]:
        buckets = defaultdict(list)
        for person, size in enumerate(groupSizes):
            buckets[size].append(person)
        return [people[i:i + size]
                for size, people in buckets.items()
                for i in range(0, len(people), size)]

for gs, _ in examples:
    print(is_valid(gs, SolutionBucket().groupThePeople(gs)))

True
True


**Độ phức tạp Cách 2:** `O(n)` thời gian (mỗi người được thêm vào bucket một lần và được copy vào đúng một lát cắt), `O(n)` bộ nhớ.

### Cách 3 — Tham lam "đổ đầy rồi chốt" (một lượt)

**Từng đoạn:**
1. `open_group = defaultdict(list)`.
2. Duyệt `(person, size)`: `open_group[size].append(person)`.
3. Nếu đủ `size` người: `res.append(open_group[size])` rồi **gán** `open_group[size] = []` (không dùng `clear()`).

In [12]:
def group_greedy(groupSizes):
    open_group = defaultdict(list)
    res = []
    for person, size in enumerate(groupSizes):
        open_group[size].append(person)
        if len(open_group[size]) == size:
            res.append(open_group[size])
            open_group[size] = []          # mở nhóm mới (list mới)
    return res

for gs, _ in examples:
    out = group_greedy(gs)
    print(out, "| hợp lệ:", is_valid(gs, out))

[[0, 1, 2], [5], [3, 4, 6]] | hợp lệ: True
[[1], [2, 3, 4], [0, 5]] | hợp lệ: True


In [13]:
class SolutionGreedy:
    def groupThePeople(self, groupSizes: List[int]) -> List[List[int]]:
        open_group = defaultdict(list)
        res = []
        for person, size in enumerate(groupSizes):
            g = open_group[size]
            g.append(person)
            if len(g) == size:
                res.append(g)
                del open_group[size]       # lần sau defaultdict tự tạo list mới
        return res

for gs, _ in examples:
    print(SolutionGreedy().groupThePeople(gs))

[[0, 1, 2], [5], [3, 4, 6]]
[[1], [2, 3, 4], [0, 5]]


**Độ phức tạp Cách 3:** `O(n)` thời gian, `O(n)` bộ nhớ. Ưu điểm: chỉ một vòng lặp, kết quả theo thứ tự "nhóm nào đủ người trước thì xuất hiện trước".

### Cách 4 — Sắp xếp ID theo kích thước rồi cắt tuần tự

**Ý tưởng:** sau khi sắp xếp, người cùng kích thước đứng liền nhau. Duyệt từ trái sang: tại vị trí `i`, lấy `s = groupSizes[ids[i]]`, nhóm là `ids[i:i+s]`, nhảy `i += s`.

In [14]:
def group_sort(groupSizes):
    ids = sorted(range(len(groupSizes)), key=lambda p: groupSizes[p])
    res, i = [], 0
    while i < len(ids):
        s = groupSizes[ids[i]]
        res.append(ids[i:i + s])
        i += s
    return res

for gs, _ in examples:
    out = group_sort(gs)
    print(out, "| hợp lệ:", is_valid(gs, out))

[[5], [0, 1, 2], [3, 4, 6]] | hợp lệ: True
[[1], [0, 5], [2, 3, 4]] | hợp lệ: True


Vì sao nhảy `i += s` không bao giờ "cắt ngang" hai kích thước khác nhau? Vì khối các người có kích thước `s` có độ dài `c_s` chia hết cho `s`, nên các điểm cắt luôn rơi đúng ranh giới khối.

**Độ phức tạp Cách 4:** `O(n log n)` do sắp xếp (có thể `O(n)` nếu dùng counting sort vì `groupSizes[i] ≤ n`), `O(n)` bộ nhớ.

In [15]:
class SolutionSort:
    def groupThePeople(self, groupSizes: List[int]) -> List[List[int]]:
        ids = sorted(range(len(groupSizes)), key=groupSizes.__getitem__)
        res, i = [], 0
        while i < len(ids):
            s = groupSizes[ids[i]]
            res.append(ids[i:i + s])
            i += s
        return res

for gs, _ in examples:
    print(is_valid(gs, SolutionSort().groupThePeople(gs)))

True
True


## 8. TRACE từng bước

Trace Cách 3 trên **Ví dụ 1** `[3, 3, 3, 3, 3, 1, 3]`:

| person | size | open[size] sau khi thêm | đủ? | res |
|---|---|---|---|---|
| 0 | 3 | [0] | không | [] |
| 1 | 3 | [0, 1] | không | [] |
| 2 | 3 | [0, 1, 2] | **đủ** → chốt | [[0,1,2]] |
| 3 | 3 | [3] | không | [[0,1,2]] |
| 4 | 3 | [3, 4] | không | [[0,1,2]] |
| 5 | 1 | [5] | **đủ** → chốt | [[0,1,2],[5]] |
| 6 | 3 | [3, 4, 6] | **đủ** → chốt | [[0,1,2],[5],[3,4,6]] |

In [16]:
def trace_greedy(groupSizes):
    open_group = defaultdict(list)
    res = []
    for person, size in enumerate(groupSizes):
        open_group[size].append(person)
        full = len(open_group[size]) == size
        print(f"person={person}, size={size}, open[{size}]={open_group[size]}"
              + ("  -> ĐỦ, chốt nhóm" if full else ""))
        if full:
            res.append(open_group[size])
            open_group[size] = []
        print(f"    res = {res}")
    return res

out = trace_greedy(examples[0][0])
print("hợp lệ:", is_valid(examples[0][0], out))

person=0, size=3, open[3]=[0]
    res = []
person=1, size=3, open[3]=[0, 1]
    res = []
person=2, size=3, open[3]=[0, 1, 2]  -> ĐỦ, chốt nhóm
    res = [[0, 1, 2]]
person=3, size=3, open[3]=[3]
    res = [[0, 1, 2]]
person=4, size=3, open[3]=[3, 4]
    res = [[0, 1, 2]]
person=5, size=1, open[1]=[5]  -> ĐỦ, chốt nhóm
    res = [[0, 1, 2], [5]]
person=6, size=3, open[3]=[3, 4, 6]  -> ĐỦ, chốt nhóm
    res = [[0, 1, 2], [5], [3, 4, 6]]
hợp lệ: True


In [17]:
def trace_sort(groupSizes):
    ids = sorted(range(len(groupSizes)), key=lambda p: groupSizes[p])
    print("ids sắp theo size:", ids, "| size:", [groupSizes[p] for p in ids])
    res, i = [], 0
    while i < len(ids):
        s = groupSizes[ids[i]]
        print(f"i={i}: size={s}, lấy ids[{i}:{i + s}] = {ids[i:i + s]}")
        res.append(ids[i:i + s])
        i += s
    return res

print(trace_sort(examples[1][0]))

ids sắp theo size: [1, 0, 5, 2, 3, 4] | size: [1, 2, 2, 3, 3, 3]
i=0: size=1, lấy ids[0:1] = [1]
i=1: size=2, lấy ids[1:3] = [0, 5]
i=3: size=3, lấy ids[3:6] = [2, 3, 4]
[[1], [0, 5], [2, 3, 4]]


## 9. Code nộp LeetCode

Chọn Cách 3 (tham lam một lượt): `O(n)`, ngắn gọn, đúng chữ ký `groupThePeople(self, groupSizes)`.

In [18]:
class Solution:
    def groupThePeople(self, groupSizes: List[int]) -> List[List[int]]:
        open_group = defaultdict(list)   # size -> nhóm đang mở
        res = []
        for person, size in enumerate(groupSizes):
            open_group[size].append(person)
            if len(open_group[size]) == size:
                res.append(open_group[size])
                open_group[size] = []    # list MỚI, không dùng clear()
        return res

for gs, _ in examples:
    out = Solution().groupThePeople(gs)
    print(out, "| hợp lệ:", is_valid(gs, out))

[[0, 1, 2], [5], [3, 4, 6]] | hợp lệ: True
[[1], [2, 3, 4], [0, 5]] | hợp lệ: True


## 10. Vì sao thuật toán đúng

1. **Mỗi người xuất hiện đúng một lần:** người `i` được `append` vào đúng một nhóm mở; nhóm đó sau này được chốt nguyên vẹn vào `res` (và không bao giờ bị sửa nữa vì ta gán list mới).
2. **Mỗi nhóm có đúng kích thước:** nhóm của kích thước `s` chỉ chứa người có `groupSizes = s`, và được chốt đúng khi đạt `s` người.
3. **Không sót người:** gọi `c_s` = số người kích thước `s`. Đề đảm bảo có đáp án ⇒ các người kích thước `s` được chia thành các nhóm cỡ `s` ⇒ `c_s` chia hết cho `s`. Nhóm mở của `s` được chốt mỗi khi đủ `s` người ⇒ sau `c_s` lần thêm, đã chốt đúng `c_s / s` nhóm và nhóm mở rỗng.
4. Vì các kích thước khác nhau không thể chung nhóm, bất kỳ cách chia nào thoả (1)(2) đều là đáp án hợp lệ — không cần "tối ưu" gì thêm.

## 11. Phân tích Time / Space

Gọi `n = len(groupSizes)`.

| Cách | Time | Space (ngoài output) |
|---|---|---|
| 1. Brute force | `O(n²)` | `O(n)` (`used`) |
| 2. Bucket + cắt | `O(n)` | `O(n)` |
| 3. Tham lam một lượt | `O(n)` | `O(n)` (worst khi nhiều nhóm mở cùng lúc) |
| 4. Sắp xếp + cắt | `O(n log n)` | `O(n)` |

Output luôn chiếm `O(n)` (tổng độ dài các nhóm bằng `n`).

## 12. Bộ test

Vì nhiều đáp án đúng, mọi test dùng `is_valid`. Ngoài ra kiểm tra: số nhóm đúng bằng `Σ c_s / s` (mọi đáp án có cùng số nhóm), và so sánh **sau khi chuẩn hoá** (sort) với các cách khác cho dữ liệu chỉ có một đáp án.

In [19]:
ALL = {
    "brute": group_brute, "bucket": group_bucket, "greedy": group_greedy, "sort": group_sort,
    "SolBucket": lambda g: SolutionBucket().groupThePeople(g),
    "SolGreedy": lambda g: SolutionGreedy().groupThePeople(g),
    "SolSort": lambda g: SolutionSort().groupThePeople(g),
    "Solution": lambda g: Solution().groupThePeople(g),
}

def expected_num_groups(groupSizes):
    return sum(c // s for s, c in Counter(groupSizes).items())

def normalize(groups):
    return sorted(sorted(g) for g in groups)

fixed = [
    [3, 3, 3, 3, 3, 1, 3],
    [2, 1, 3, 3, 3, 2],
    [1],                       # một người
    [1, 1, 1, 1],              # toàn nhóm 1 người
    [4, 4, 4, 4],              # một nhóm gồm tất cả
    [2, 2, 2, 2, 2, 2],
    [1, 2, 3, 2, 3, 3, 1],
]
for gs in fixed:
    for name, f in ALL.items():
        out = f(gs)
        assert is_valid(gs, out), (name, gs, out)
        assert len(out) == expected_num_groups(gs), (name, gs, out)
# dữ liệu có đáp án duy nhất (mỗi size có đúng s người) -> so sau chuẩn hoá
uniq = [2, 1, 3, 3, 3, 2]
assert len({str(normalize(f(uniq))) for f in ALL.values()}) == 1
print("Test cố định: qua", len(fixed), "case x", len(ALL), "cách")

Test cố định: qua 7 case x 8 cách


In [20]:
def random_group_sizes(n, rng):
    """Sinh groupSizes hợp lệ: xáo trộn người rồi chia thành các nhóm có kích thước ngẫu nhiên."""
    people = list(range(n))
    rng.shuffle(people)
    gs = [0] * n
    i = 0
    while i < n:
        s = rng.randint(1, n - i)
        if rng.random() < 0.7:
            s = min(s, rng.randint(1, 4))      # ưu tiên nhóm nhỏ để có nhiều nhóm
        for p in people[i:i + s]:
            gs[p] = s
        i += s
    return gs

rng = random.Random(1282)
for _ in range(1000):
    n = rng.randint(1, 30)
    gs = random_group_sizes(n, rng)
    for name, f in ALL.items():
        out = f(gs)
        assert is_valid(gs, out), (name, gs, out)
        assert len(out) == expected_num_groups(gs)
big = random_group_sizes(500, rng)
t = time.perf_counter()
assert is_valid(big, Solution().groupThePeople(big))
print("1000 test ngẫu nhiên + n = 500: mọi cách cho đáp án hợp lệ,",
      f"Solution n=500 mất {time.perf_counter() - t:.4f}s")

1000 test ngẫu nhiên + n = 500: mọi cách cho đáp án hợp lệ, Solution n=500 mất 0.0002s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Gom bucket nhưng quên cắt thành nhóm cỡ `s`

In [21]:
def wrong_no_split(groupSizes):
    buckets = defaultdict(list)
    for person, size in enumerate(groupSizes):
        buckets[size].append(person)
    return list(buckets.values())        # SAI: bucket có thể chứa nhiều nhóm

gs = examples[0][0]
out = wrong_no_split(gs)
print("Code sai:", out, "| hợp lệ?", is_valid(gs, out))

Code sai: [[0, 1, 2, 3, 4, 6], [5]] | hợp lệ? False


6 người cần nhóm cỡ 3 bị gộp thành **một** nhóm 6 người. **Code đúng:** cắt `people[i:i+size]` (Cách 2).

In [22]:
out = group_bucket(gs)
print("Code đúng:", out, "| hợp lệ?", is_valid(gs, out))

Code đúng: [[0, 1, 2], [3, 4, 6], [5]] | hợp lệ? True


### Lỗi 2 — Chốt nhóm bằng `res.append(cur)` rồi `cur.clear()`

In [23]:
def wrong_clear(groupSizes):
    open_group = defaultdict(list)
    res = []
    for person, size in enumerate(groupSizes):
        open_group[size].append(person)
        if len(open_group[size]) == size:
            res.append(open_group[size])
            open_group[size].clear()       # SAI: xoá luôn nhóm vừa đưa vào res
    return res

out = wrong_clear(gs)
print("Code sai:", out, "| hợp lệ?", is_valid(gs, out))

Code sai: [[], [], []] | hợp lệ? False


Mọi nhóm trong `res` đều là **cùng một list** (sau đó bị xoá/đổ lại). **Code đúng:** `open_group[size] = []` hoặc `res.append(open_group[size][:])` rồi `clear()`.

### Lỗi 3 — Kiểm tra đáp án bằng so sánh cứng với output mẫu

In [24]:
gs, sample = examples[0]
out = Solution().groupThePeople(gs)
print("Output của ta :", out)
print("Output mẫu    :", sample)
print("So sánh cứng  :", out == sample, "<- kết luận SAI rằng code sai")
print("is_valid      :", is_valid(gs, out))
print("So sau chuẩn hoá:", normalize(out) == normalize(sample))

Output của ta : [[0, 1, 2], [5], [3, 4, 6]]
Output mẫu    : [[5], [0, 1, 2], [3, 4, 6]]
So sánh cứng  : False <- kết luận SAI rằng code sai
is_valid      : True
So sau chuẩn hoá: True


Thứ tự nhóm khác nhau nhưng cả hai đều đúng. **Cách đúng:** dùng `is_valid`, hoặc so sánh sau khi chuẩn hoá (`normalize`) khi biết đáp án là duy nhất (sai khi có nhiều cách chia thật sự, ví dụ 6 người cỡ 3 có 10 cách chia).

### Lỗi 4 — Điều kiện chốt nhóm sai (`>` thay vì `==`)

In [25]:
def wrong_compare(groupSizes):
    open_group = defaultdict(list)
    res = []
    for person, size in enumerate(groupSizes):
        if len(open_group[size]) > size:          # SAI: chốt khi đã quá
            res.append(open_group[size])
            open_group[size] = []
        open_group[size].append(person)
    res.extend(g for g in open_group.values() if g)
    return res

gs = [2, 2, 2, 2, 1]
out = wrong_compare(gs)
print("Code sai:", out, "| hợp lệ?", is_valid(gs, out))
print("Code đúng:", Solution().groupThePeople(gs))

Code sai: [[0, 1, 2], [3], [4]] | hợp lệ? False
Code đúng: [[0, 1], [2, 3], [4]]


**Code đúng:** thêm người trước, rồi kiểm tra `len(...) == size` ngay sau khi thêm (như `Solution`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Gom phần tử theo một thuộc tính" ⇒ **`defaultdict(list)`** (bucket). Gặp ở LC 49 *Group Anagrams*, LC 1282, LC 2491…
- "Chia thành các nhóm kích thước cố định" ⇒ **cắt lát** `lst[i:i+k]` hoặc **đổ đầy rồi chốt**.
- Đề nói "**return any of them**" ⇒ chuẩn bị hàm `is_valid` để tự test.
- Luôn nhớ: `append` một list là thêm **tham chiếu**. Mở nhóm mới bằng `= []`.
- `n ≤ 500` ⇒ cả `O(n²)` cũng qua, nhưng `O(n)` dễ viết không kém.

## 15. Biến thể của bài

1. **Không đảm bảo có đáp án:** trước tiên kiểm tra `c_s % s == 0` với mọi `s`; nếu không thì trả `[]`.
2. **Đếm số cách chia khác nhau:** với mỗi kích thước `s` có `c_s` người, số cách chia thành `c_s/s` nhóm không thứ tự là `c_s! / ((s!)^(c_s/s) · (c_s/s)!)`; nhân các kích thước lại (xem Bài tập 5).
3. **Yêu cầu đáp án nhỏ nhất theo thứ tự từ điển:** cách bucket theo thứ tự ID tăng rồi sort danh sách nhóm theo phần tử đầu.
4. **Nhóm theo khoảng kích thước** (mỗi người chấp nhận nhóm từ `a_i` đến `b_i` người) — khó hơn nhiều, thường cần tham lam + sắp xếp.
5. Bài liên quan: LC 49 *Group Anagrams*, LC 846 *Hand of Straights*, LC 1296 *Divide Array in Sets of K Consecutive Numbers*.

In [26]:
def group_or_empty(groupSizes):
    cnt = Counter(groupSizes)
    if any(c % s for s, c in cnt.items()):
        return []                       # không có cách chia hợp lệ
    return Solution().groupThePeople(groupSizes)

print(group_or_empty([2, 2, 1]), group_or_empty([2, 2, 2]), group_or_empty([3, 3, 3]))

[[0, 1], [2]] [] [[0, 1, 2]]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | mỗi người chưa xếp, quét tìm bạn cùng cỡ | `O(n²)` | `O(n)` | hiểu bài, `n` nhỏ |
| 2. Bucket + cắt | gom theo size, cắt lát cỡ `s` | `O(n)` | `O(n)` | rõ ràng, dễ giải thích |
| 3. Đổ đầy rồi chốt | nhóm mở cho từng size, đủ thì chốt | `O(n)` | `O(n)` | **nộp bài**, một lượt |
| 4. Sắp xếp + cắt | sort ID theo size, nhảy `i += s` | `O(n log n)` | `O(n)` | khi muốn tránh dict |

## 17. Bài tập vận dụng

**Bài 1 (đoán output):** Kết quả của `Solution().groupThePeople([1, 1, 1])` và `Solution().groupThePeople([2, 2, 1, 2, 2])` là gì (theo đúng code ở mục 9)?

**Bài 2 (tính tay):** Với `groupSizes = [2, 2, 2, 2, 1, 3, 3, 3]`, có bao nhiêu nhóm? Liệt kê một đáp án.

**Bài 3 (điền TODO):** Hoàn thiện `is_valid_todo(groupSizes, groups)` kiểm tra một đáp án (không nhìn mục 4).

**Bài 4 (tự viết lại):** Viết lại Cách 4 nhưng dùng **counting sort** thay cho `sorted` (vì `1 ≤ groupSizes[i] ≤ n`), đạt `O(n)`.

**Bài 5 (mini challenge):** Viết `count_ways(groupSizes)` đếm **số cách chia khác nhau** (nhóm không có thứ tự, người trong nhóm không có thứ tự). Kiểm chứng bằng vét cạn với `n` nhỏ.

In [27]:
def is_valid_todo(groupSizes, groups):
    n = len(groupSizes)
    # TODO 1: kiểm tra mỗi người 0..n-1 xuất hiện đúng 1 lần
    # TODO 2: kiểm tra với mỗi nhóm g, mọi p trong g có groupSizes[p] == len(g)
    pass

def group_counting_sort_todo(groupSizes):
    n = len(groupSizes)
    by_size = [[] for _ in range(n + 1)]
    # TODO: đưa người vào by_size[size], rồi nối lại và cắt như Cách 4
    pass

# Lưu ý: cell này chỉ định nghĩa hàm, không gọi.

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Chạy tay đúng thứ tự `enumerate`. (2) Kích thước 1 chốt ngay khi gặp. (3) Ở dãy thứ hai: người 0, 1 chốt `[0, 1]`; người 2 chốt `[2]`; người 3, 4 chốt `[3, 4]`.

**Bài 2.** (1) Đếm: `c_2 = 4`, `c_1 = 1`, `c_3 = 3`. (2) Số nhóm `= 4/2 + 1/1 + 3/3`. (3) Một đáp án: `[[0,1],[2,3],[4],[5,6,7]]`.

**Bài 3.** (1) Mảng `seen` đếm số lần xuất hiện. (2) Kiểm tra chỉ số hợp lệ trước khi truy cập. (3) Cuối cùng `all(c == 1 for c in seen)`.

**Bài 4.** (1) `by_size[s].append(p)` theo thứ tự `p` tăng. (2) `ids = [p for s in range(1, n+1) for p in by_size[s]]`. (3) Cắt như Cách 4 hoặc cắt trực tiếp từng `by_size[s]`.

**Bài 5.** (1) Các kích thước độc lập ⇒ nhân số cách. (2) Chia `c` người thành `k = c/s` nhóm cỡ `s` không thứ tự: `c! / ((s!)^k · k!)`. (3) Vét cạn: đệ quy luôn đặt người nhỏ nhất chưa xếp vào nhóm mới, chọn `s−1` bạn từ những người cùng cỡ còn lại.

## 19. Đáp án tham khảo

In [28]:
from itertools import combinations

# Bài 1
assert Solution().groupThePeople([1, 1, 1]) == [[0], [1], [2]]
assert Solution().groupThePeople([2, 2, 1, 2, 2]) == [[0, 1], [2], [3, 4]]

# Bài 2
gs2 = [2, 2, 2, 2, 1, 3, 3, 3]
assert expected_num_groups(gs2) == 4
assert is_valid(gs2, [[0, 1], [2, 3], [4], [5, 6, 7]])

# Bài 3
def is_valid_sol(groupSizes, groups):
    n = len(groupSizes)
    seen = [0] * n
    for g in groups:
        for p in g:
            if not 0 <= p < n or groupSizes[p] != len(g):
                return False
            seen[p] += 1
    return all(c == 1 for c in seen)

# Bài 4
def group_counting_sort_sol(groupSizes):
    n = len(groupSizes)
    by_size = [[] for _ in range(n + 1)]
    for p, s in enumerate(groupSizes):
        by_size[s].append(p)
    res = []
    for s in range(1, n + 1):
        people = by_size[s]
        for i in range(0, len(people), s):
            res.append(people[i:i + s])
    return res

# Bài 5
def count_ways_sol(groupSizes):
    total = 1
    for s, c in Counter(groupSizes).items():
        k = c // s
        total *= factorial(c) // (factorial(s) ** k * factorial(k))
    return total

def count_ways_brute(groupSizes):
    def go(remaining):
        if not remaining:
            return 1
        first = min(remaining)
        s = groupSizes[first]
        mates = [p for p in remaining if p != first and groupSizes[p] == s]
        ways = 0
        for combo in combinations(mates, s - 1):
            ways += go(remaining - {first, *combo})
        return ways
    return go(frozenset(range(len(groupSizes))))

assert count_ways_sol([3, 3, 3, 3, 3, 1, 3]) == 10      # 6 người cỡ 3: 10 cách
rng = random.Random(5)
for _ in range(300):
    gs = random_group_sizes(rng.randint(1, 9), rng)
    assert is_valid_sol(gs, Solution().groupThePeople(gs))
    assert is_valid_sol(gs, group_counting_sort_sol(gs))
    assert count_ways_sol(gs) == count_ways_brute(gs)
assert not is_valid_sol([3, 3, 3, 3, 3, 1, 3], [[5], [0, 1, 2, 3], [4, 6]])
assert not is_valid_sol([1, 1], [[0], [0]])
print("Đáp án Bài 1–5: mọi assert đều qua")

Đáp án Bài 1–5: mọi assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "đổ đầy rồi chốt":**

```python
open_group = defaultdict(list)
res = []
for i, key in enumerate(arr):
    open_group[key].append(i)
    if len(open_group[key]) == capacity(key):
        res.append(open_group[key])
        open_group[key] = []        # list mới!
```

**Template bucket + cắt:**

```python
buckets = defaultdict(list)
for i, s in enumerate(arr):
    buckets[s].append(i)
res = [b[i:i+s] for s, b in buckets.items() for i in range(0, len(b), s)]
```

**Checklist trước khi submit:**
- [ ] Mỗi nhóm có đúng `groupSizes[p]` người.
- [ ] Mở nhóm mới bằng `= []`, không `clear()` list đã đưa vào kết quả.
- [ ] Kiểm tra `==` ngay **sau** khi thêm người.
- [ ] Test bằng `is_valid`, không so sánh cứng với output mẫu.
- [ ] Đúng chữ ký: `groupThePeople(self, groupSizes: List[int]) -> List[List[int]]`.